In [3]:
import numpy as np
import pandas as pd

In [4]:
df=pd.read_csv("new_unicorn_Companies.csv")

 NULL HANDLING

In [5]:
df.isnull().sum()

df["city"] = df["city"].fillna("Unknown")

industry_funding_median = df.groupby("industry")["funding"].median()
df["funding"] = df["funding"].fillna(industry_funding_median)

df["investors"] = df["investors"].fillna("Not disclosed")


UNIT NORMALISATION

In [6]:
df["valuation_B"] = (df["valuation"] / 1e9).round(2)
df["funding_B"] = (df["funding"] / 1e9).round(3)
df=df.drop(columns=["valuation","funding"])

INDUSTRY NAME STANDARDISATION

In [7]:
df["industry"].value_counts()
df["industry"]=(df["industry"].str.strip().str.title())

DERIVED FEATURES

In [8]:
#years to become a unicorn
df["years_to_unicorn"] = df["year_joined"] - df["year_founded"]

# FTV (Funding-to-Valuation)
df["ftv_ratio"] = np.where(df["valuation_B"] > 0,(df["funding_B"] / df["valuation_B"]).round(4),np.nan)

# VTF (Valuation-to-Funding)
df["vtf_ratio"] = np.where(df["funding_B"] > 0,(df["valuation_B"] / df["funding_B"]).round(2),np.nan)
   
# Valuation tier
bins   = [0, 1, 5, 10, 1000]
labels = ["Emerging ($1B)", "Mid ($1-5B)", "Large ($5-10B)", "Mega ($10B+)"]
df["valuation_tier"] = pd.cut(df["valuation_B"], bins=bins, labels=labels)

# Splitting the raw comma-separated investors string
def parse_investors(s: str):
    if not s or s == "Not disclosed":
        return []
    return [i.strip() for i in s.split(",") if i.strip()]

df["investor_list"]  = df["investors"].apply(parse_investors)
df["investor_count"] = df["investor_list"].apply(len)


UNDERSTANDING THE OUTLIER

In [9]:
Q1  = df["valuation_B"].quantile(0.25)
Q3  = df["valuation_B"].quantile(0.75)
IQR = Q3 - Q1
valuation_outlier = ((df["valuation_B"] < (Q1 - 1.5 * IQR)) |(df["valuation_B"] > (Q3 + 1.5 * IQR)))
valuation_outlier.value_counts()

valuation_B
False    972
True     102
Name: count, dtype: int64

In [10]:
df.to_csv("cleaned_unicorns.csv",index=False)

In [11]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

INPUT_CSV  = "cleaned_unicorns.csv"
OUTPUT_DIR = "dim_tables"

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT_CSV)

df = df.rename(columns={"vtf_ratio": "valuation_to_funding_ratio"})

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

dim_industry = (df[["industry"]]
                 .drop_duplicates()
                 .reset_index(drop=True))
dim_industry.index.name = "industry_id"
dim_industry = dim_industry.reset_index()
dim_industry["industry_id"] = dim_industry["industry_id"] + 1

dim_industry.to_csv(os.path.join(OUTPUT_DIR, "dim_industry.csv"), index=False)
print(f"[dim_industry] {len(dim_industry)} rows")

dim_geography = (df[["city", "country", "continent"]]
                  .drop_duplicates()
                  .reset_index(drop=True))
dim_geography.index.name = "geo_id"
dim_geography = dim_geography.reset_index()
dim_geography["geo_id"] = dim_geography["geo_id"] + 1

dim_geography.to_csv(os.path.join(OUTPUT_DIR, "dim_geography.csv"), index=False)
print(f"[dim_geography] {len(dim_geography)} rows")

investor_rows = []
for _, row in df.iterrows():
    investors_str = row["investors"]
    if pd.isna(investors_str) or investors_str == "Not disclosed":
        continue
    for inv in str(investors_str).split(","):
        inv = inv.strip()
        if inv:
            investor_rows.append({
                "company_id": row["company_id"],
                "company": row["company"],
                "investor": inv
            })

dim_investor = pd.DataFrame(investor_rows)
dim_investor.to_csv(os.path.join(OUTPUT_DIR, "dim_investor.csv"), index=False)
print(f"[dim_investor] {len(dim_investor)} rows | "
      f"{dim_investor['investor'].nunique()} unique investors")

fact_unicorn = df.merge(
    dim_geography[["geo_id", "city", "country", "continent"]],
    on=["city", "country", "continent"],
    how="left"
).merge(
    dim_industry[["industry_id", "industry"]],
    on="industry",
    how="left"
)

fact_cols = [
    "company_id", "company", "valuation_B", "funding_B",
    "valuation_to_funding_ratio", "ftv_ratio", "valuation_tier",
    "year_founded", "year_joined", "years_to_unicorn",
    "industry", "city", "country", "continent",
    "investors", "geo_id", "industry_id"
]
fact_unicorn = fact_unicorn[fact_cols]

fact_unicorn.to_csv(os.path.join(OUTPUT_DIR, "fact_unicorn.csv"), index=False)
print(f"[fact_unicorn] {len(fact_unicorn)} rows, {len(fact_unicorn.columns)} columns")

# Sanity check: confirm every row got a valid geo_id and industry_id
missing_geo = fact_unicorn["geo_id"].isnull().sum()
missing_ind = fact_unicorn["industry_id"].isnull().sum()
if missing_geo or missing_ind:
    print(f"⚠ WARNING: {missing_geo} rows missing geo_id, "
          f"{missing_ind} rows missing industry_id -- check for a merge-key mismatch")
else:
    print("✓ All rows matched a geo_id and industry_id -- no merge gaps")

print(f"\nAll 4 tables saved to ./{OUTPUT_DIR}/")



Loaded 1074 rows, 17 columns
[dim_industry] 15 rows
[dim_geography] 262 rows
[dim_investor] 3051 rows | 1253 unique investors
[fact_unicorn] 1074 rows, 17 columns
✓ All rows matched a geo_id and industry_id -- no merge gaps

All 4 tables saved to ./dim_tables/
